## 固定示例

最基本（也是常见）的小样本提示技术是使用固定提示示例。这样您就可以选择一条链条，对其进行评估，并避免担心生产中的额外移动部件。

模板的基本组件是： 
- examples ：要包含在最终提示中的字典示例列表。 
- example_prompt ：通过其 format_messages 方法将每个示例转换为 1 条或多条消息。一个常见的示例是将每个示例转换为一条人工消息和一条人工智能消息响应，或者一条人工消息后跟一条函数调用消息。

In [1]:
from langchain.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
)

In [2]:
examples = [
    {"input": "2+2", "output": "4"},
    {"input": "2+3", "output": "5"},
]

组装成少示例的提示模板。

In [4]:
# This is a prompt template used to format each individual example.
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

print(few_shot_prompt.format())

Human: 2+2
AI: 4
Human: 2+3
AI: 5


In [5]:
#最后，组装最终的提示并将其与模型一起使用。

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一位非常厉害的数学天才。"),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

In [6]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
openai_api_key = "EMPTY"
openai_api_base = "http://127.0.0.1:1234/v1"
chat = ChatOpenAI(
    openai_api_key=openai_api_key,
    openai_api_base=openai_api_base,
    temperature=0.7,
)

output_parser = StrOutputParser()

chain = final_prompt | chat | output_parser

chain.invoke({"input":"3的平方是多少？"})

'<think>\n嗯，用户问的是“3的平方是多少？”这个问题看起来简单，但作为数学天才，我得仔细思考一下。首先，平方就是把一个数乘以自己本身，对吧？所以3的平方应该是3×3。不过，可能用户是想确认基本的概念，或者有没有其他需要注意的地方。\n\n先回忆一下基本的数学运算规则。平方的话，无论正负数都是一样的，但这里3是正数，所以结果肯定是正的。计算的话，3乘以3等于9。不过，是不是有什么特殊情况呢？比如在某些数学领域里，平方可能有不同的定义，但通常来说，在基础数学中，平方就是乘法。\n\n再想想用户可能的意图。也许他们是在学习代数，或者刚开始学数学，所以需要确认基本概念是否正确。也有可能他们在做作业的时候遇到了这个问题，想确认答案是否正确。这时候，我应该给出明确的答案，并且解释清楚过程，确保他们理解。\n\n有没有可能用户有其他隐藏的需求？比如，他们可能在问为什么是9而不是别的数字，或者是否有其他数学上的解释？不过根据问题本身，直接回答平方就是乘法的结果即可。另外，考虑到用户之前的问题都是关于基本的算术运算，所以保持答案简洁明了比较合适。\n\n再检查一下计算是否正确。3×3确实是9，没错。所以最终的答案应该是9。同时，可能需要用简单的话解释，比如“3的平方是3乘以3等于9”。这样用户就能清楚理解步骤和结果了。\n</think>\n\n3的平方是 **9**。  \n（因为平方即一个数乘以它本身，即 $3 \\times 3 = 9$）'

In [7]:
chat.invoke(input="3的平方是多少？")

AIMessage(content='<think>\n嗯，用户问的是“3的平方是多少？”，这个问题看起来简单，但可能需要仔细考虑。首先，我得确认“平方”在数学中的定义。平方通常指的是一个数乘以它自己，也就是a² = a × a。所以，3的平方应该是3×3。\n\n不过，可能用户会有不同的理解，比如是否涉及其他数学概念，或者是否有特殊的情况需要考虑。比如，在某些上下文中，平方可能会有不同的解释，但一般来说，基本的定义应该没问题。我需要确保自己没有遗漏任何可能的陷阱或特殊情况。\n\n另外，用户可能是学生，正在学习基础的数学知识，所以需要明确回答，并且用简单易懂的语言解释。可能还要考虑到是否需要举例说明或者给出更多的例子来巩固概念。不过这个问题比较直接，不需要太多扩展。\n\n再检查一下计算是否正确，3乘以3确实是9，没错。有没有可能用户有其他意图？比如，是否在问3的平方根是多少？但问题明确说是“平方”，所以应该是求平方而不是开方。不过为了全面考虑，可以稍微提醒一下，如果用户指的是平方根的话结果会不同，但根据问题本身，应该还是9。\n\n另外，可能需要注意单位或者是否有其他隐藏的信息，但问题中没有提到任何额外的条件或上下文，所以应该直接回答即可。总结一下，答案应该是9，同时确认计算过程正确，并且解释清楚平方的定义。\n</think>\n\n3的平方是 **9**。\n\n**解析：**  \n平方表示一个数乘以它本身，即 $ 3^2 = 3 \\times 3 = 9 $。\n\n如果问题有其他上下文（如物理中的面积、坐标等），可能需要进一步分析，但根据常规数学定义，答案明确为 **9**。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 374, 'prompt_tokens': 13, 'total_tokens': 387, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'qwen/qwen3-1.7b', 'system_fingerprint': 'qwen/qwen3-1.7b', 'id': 'ch

## 动态几次提示

有时您可能希望根据输入来限制显示哪些示例。为此，您可以将 examples 替换为 example_selector 。其他组件与上面相同！回顾一下，动态几次提示模板将如下所示：

- example_selector ：负责为给定输入选择少数样本（以及它们返回的顺序）。它们实现了 BaseExampleSelector 接口。一个常见的例子是向量存储支持的 SemanticSimilarityExampleSelector

- example_prompt ：通过其 format_messages 方法将每个示例转换为 1 条或多条消息。一个常见的示例是将每个示例转换为一条人工消息和一条人工智能消息响应，或者一条人工消息后跟一条函数调用消息。

这些可以再次与其他消息和聊天模板组合以组合您的最终提示。

In [8]:
from langchain_community.embeddings.huggingface import HuggingFaceEmbeddings

embeddings_path = r"C:\Users\jerry\.cache\huggingface\hub\models--BAAI--bge-large-zh-v1.5\snapshots\79e7739b6ab944e86d6171e44d24c997fc1e0116"
embeddings = HuggingFaceEmbeddings(model_name=embeddings_path)

C:\Users\jerry\AppData\Local\Temp\ipykernel_30108\2872281868.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embeddings_path)


In [9]:
from langchain.prompts.example_selector import SemanticSimilarityExampleSelector
from langchain_community.vectorstores import Chroma
examples = [
    {"input": "2+2", "output": "4"},
    {"input": "2+3", "output": "5"},
    {"input": "2+4", "output": "6"},
    {"input": "牛对月亮说了什么？", "output": "什么都没有"},
    {
        "input": "给我写一首关于月亮的五言诗",
        "output": "月儿挂枝头，清辉洒人间。 银盘如明镜，照亮夜归人。 思绪随风舞，共赏中秋圆。",
    },
]

to_vectorize = [" ".join(example.values()) for example in examples]
vectorstore = Chroma.from_texts(to_vectorize, embeddings, metadatas=examples)

ImportError: Could not import chromadb python package. Please install it with `pip install chromadb`.

In [11]:
example_selector = SemanticSimilarityExampleSelector(
    vectorstore=vectorstore,
    k=2,
)

# The prompt template will load examples by passing the input do the `select_examples` method
example_selector.select_examples({"input": "对牛弹琴"})

[{'input': '牛对月亮说了什么？', 'output': '什么都没有'}, {'input': '2+2', 'output': '4'}]

In [12]:
from langchain.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
)

# Define the few-shot prompt.
few_shot_prompt = FewShotChatMessagePromptTemplate(
    # The input variables select the values to pass to the example_selector
    input_variables=["input"],
    example_selector=example_selector,
    # Define how each example will be formatted.
    # In this case, each example will become 2 messages:
    # 1 human, and 1 AI
    example_prompt=ChatPromptTemplate.from_messages(
        [("human", "{input}"), ("ai", "{output}")]
    ),
)

In [13]:
few_shot_prompt.format(input="What's 3+3?")

'Human: 2+3\nAI: 5\nHuman: 2+4\nAI: 6'

In [18]:
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一位非常厉害的数学天才。"),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

In [19]:
print(final_prompt.format(input="3+5是多少？"))

System: 你是一位非常厉害的数学天才。
Human: 2+3
AI: 5
Human: 2+4
AI: 6
Human: 3+5是多少？


In [21]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
openai_api_key = "EMPTY"
openai_api_base = "http://127.0.0.1:1234/v1"
chat = ChatOpenAI(
    openai_api_key=openai_api_key,
    openai_api_base=openai_api_base,
    temperature=0.7,
)

output_parser = StrOutputParser()

chain = final_prompt | chat | output_parser

chain.invoke({"input":"3+5是多少？"})

'8'